In [1]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid
from torch.nn.functional import interpolate

from datasets import load_dataset
from inpaintingStart import *
import argparse

from PIL import Image
import gradio as gr
from imagenet_en_cn import IMAGENET_1K_CLASSES
from omegaconf import OmegaConf

import torch
from transformers import T5EncoderModel, AutoTokenizer

from pixelflow.scheduling_pixelflow import PixelFlowScheduler
from pixelflow.pipeline_pixelflow import PixelFlowPipeline
from pixelflow.utils import config as config_utils
from pixelflow.utils.misc import seed_everything

from sampler import PixelFlowPipeline2
from sampler2 import PixelFlowPipeline3

import matplotlib.pyplot as plt
import imageio
import math
import copy
import os
from pathlib import Path
from einops import rearrange
# ds = load_dataset("ILSVRC/imagenet-1k")

/home/ruy45/anaconda3/envs/pixelflow/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
num_stages = 4 # config.scheduler.num_stages
inference_each_step = 10
class_label = 10
NUM_EXAMPLES = 4
resolution = 256
num_Langevin = 100
lr_base = 1e-4
lr_min_ratio = 1e-2
sigma_n =0.05
proj = True
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
output_dir = './pretrained_models/c2img'
A_opr = get_operator("inpainting", mask_type='box', mask_len_range=(80, 160), mask_prob_range=None, resolution=resolution, device=device, sigma=sigma_n)
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
config = OmegaConf.load(f"{output_dir}/config.yaml")
model = config_utils.instantiate_from_config(config.model).to(device)
print(f"Num of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
ckpt = torch.load(f"{output_dir}/model.pt", map_location="cpu", weights_only=False)
model.load_state_dict(ckpt, strict=True)
model.eval()
text_encoder = None
tokenizer = None
scheduler = PixelFlowScheduler(config.scheduler.num_train_timesteps, num_stages=config.scheduler.num_stages, gamma=-1/3)
scheduler_copy = copy.deepcopy(scheduler)
sampler_pipeline = PixelFlowPipeline3(scheduler_copy,model,text_encoder=text_encoder,tokenizer=tokenizer, max_token_length=512,)

Starting Initialization...
Num of parameters: 676611120


In [3]:
# noises = []
# for noise_idx in range(1, config.scheduler.num_stages):
#     noises.append(np.load(os.path.join(result_dir, f"noise_stage_{noise_idx}.npy")))
# noises[0].shape, noises[1].shape,noises[2].shape,

In [4]:
import os
x0s = []
seed_everything(0)
result_dir = f"./latents_stages{config.scheduler.num_stages}_Time_steps{10}/"
noises = []
for noise_idx in range(1, config.scheduler.num_stages):
    noises.append(torch.tensor(np.load(os.path.join(result_dir, f"noise_stage_{noise_idx}.npy")), dtype=torch.float32, device=device))

latent_files = sorted([os.path.join(result_dir, "mid_states",i) for i in os.listdir(result_dir +"./mid_states")])
target = np.load(os.path.join(result_dir, "target.npy"))
for latent_file in latent_files:
    xt = torch.from_numpy(np.load(latent_file)).to(device)
    T = torch.tensor(float(latent_file.split("_")[-2]), dtype = torch.float32, device = device)
    print(T)
    stage_idx = int(latent_file.split("_")[-3])
    with torch.autocast("cuda", dtype=torch.bfloat16), torch.no_grad():
        samples = sampler_pipeline(prompt=[class_label] * NUM_EXAMPLES,
                                        height=resolution,
                                        width=resolution,
                                        num_inference_steps=10,
                                        guidance_scale=0.0,
                                        num_images_per_prompt=1,
                                        device=device,
                                        shift=1.0,
                                        use_ode_dopri5=False,
                                        xt = xt,          
                                        start_stage = stage_idx,             
                                        start_T = T,
                                        noises = noises,          
                                        normalized_x0_hat = False)
        x0s.append(samples.cpu().float().numpy())
        


tensor(0., device='cuda:1')
tensor(19.4250, device='cuda:1')
tensor(0.1110, device='cuda:1', dtype=torch.float64)
Timesteps: tensor([ 19.4250,  38.8500,  58.2750,  77.7000,  97.1250, 116.5500, 135.9750,
        155.4000, 174.8250], device='cuda:1', dtype=torch.float64)
ts: tensor([0.1110, 0.2220, 0.3330, 0.4440, 0.5550, 0.6660, 0.7770, 0.8880, 0.9990,
        1.0000], device='cuda:1', dtype=torch.float64)
tensor(38.8500, device='cuda:1')
tensor(0.2220, device='cuda:1', dtype=torch.float64)
Timesteps: tensor([ 38.8500,  58.2750,  77.7000,  97.1250, 116.5500, 135.9750, 155.4000,
        174.8250], device='cuda:1', dtype=torch.float64)
ts: tensor([0.2220, 0.3330, 0.4440, 0.5550, 0.6660, 0.7770, 0.8880, 0.9990, 1.0000],
       device='cuda:1', dtype=torch.float64)
tensor(58.2750, device='cuda:1')
tensor(0.3330, device='cuda:1', dtype=torch.float64)
Timesteps: tensor([ 58.2750,  77.7000,  97.1250, 116.5500, 135.9750, 155.4000, 174.8250],
       device='cuda:1', dtype=torch.float64)
ts: tens

In [6]:
plot_idx = 0
T_list = np.array([float(latent_file.split("_")[-2]) for latent_file in latent_files])


In [7]:
len(target)

4

In [11]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter

plot_idx = 0
diff = np.array([x[plot_idx] - target[plot_idx] for x in x0s])
abs_err = np.abs(diff)
sq_err = diff ** 2

mae = [float(x.mean()) for x in abs_err]
mse = [float(x.mean()) for x in sq_err]
# ===== 你已有的工具函数（原样保留） =====
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().float().cpu().numpy()
    return np.asarray(x)

def to_display_image(x, batch_idx=0):
    x = to_numpy(x)

    # 如果有 batch 维，取指定 batch
    if x.ndim == 4:
        if not (0 <= batch_idx < x.shape[0]):
            raise IndexError(f"batch_idx={batch_idx} out of range for batch size {x.shape[0]}")
        x = x[batch_idx]

    # CHW -> HWC
    if x.ndim == 3 and x.shape[0] in (1, 3):
        x = np.transpose(x, (1, 2, 0))

    # 如果是单通道 HWC，压成 HW
    if x.ndim == 3 and x.shape[-1] == 1:
        x = x[..., 0]

    return x

def make_error_map(x_img, target_img):
    """
    返回 2D heatmap（HxW）
    - 灰度: abs(x-target)
    - RGB/多通道: 按通道做 L2 norm（也可以改成 mean abs）
    """
    x = x_img.astype(np.float32)
    y = target_img.astype(np.float32)

    if x.shape != y.shape:
        raise ValueError(f"Shape mismatch: x {x.shape}, target {y.shape}")

    diff = x - y

    if diff.ndim == 2:
        err = np.abs(diff)
    elif diff.ndim == 3:
        # 通道维在最后，做每像素 L2 误差
        err = np.linalg.norm(diff, axis=-1)
    else:
        raise ValueError(f"Unsupported ndim for visualization: {diff.ndim}")

    return err

def normalize_for_display(img, assume_image_range=True):
    """
    显示用：
    - 如果是正常图像（大概率在 [0,1] 或 [0,255]），直接处理
    - 如果是 latent（范围乱），做 min-max 归一化显示
    """
    img = img.astype(np.float32)

    if img.ndim == 2:
        if assume_image_range:
            if img.max() > 1.5:
                img_disp = np.clip(img / 255.0, 0, 1)
            else:
                img_disp = np.clip(img, 0, 1)
        else:
            mn, mx = img.min(), img.max()
            img_disp = (img - mn) / (mx - mn + 1e-8)
        return img_disp

    elif img.ndim == 3:
        if assume_image_range:
            if img.max() > 1.5:
                img_disp = np.clip(img / 255.0, 0, 1)
            else:
                img_disp = np.clip(img, 0, 1)
        else:
            mn, mx = img.min(), img.max()
            img_disp = (img - mn) / (mx - mn + 1e-8)
        return img_disp

    else:
        raise ValueError(f"Unsupported image ndim: {img.ndim}")


# ===== 新增：时间格式化函数 =====
def to_scalar(x):
    if isinstance(x, torch.Tensor):
        return float(x.detach().item())
    return float(x)

def format_T(x, int_digits=4, frac_digits=3):
    # 例如 872.999 -> 0872.999（整数位固定宽度，不够补0）
    width = int_digits + 1 + frac_digits
    return f"{float(x):0{width}.{frac_digits}f}"


# =========================
# 2) 预处理 x0s 和 target
# =========================
# 你需要额外提供：
# T_list = [...]   # 长度与 x0s 相同，每一项对应一帧的 T
# 例如在生成 x0s 时同步 append: T_list.append(float(T.item()))

x0s_img = [to_display_image(x, batch_idx = plot_idx) for x in x0s]
target_img = to_display_image(target, plot_idx)

assert len(T_list) == len(x0s_img), f"T_list 长度({len(T_list)})必须和 x0s({len(x0s_img)})一致"

# 计算 error maps
error_maps = [make_error_map(x, target_img) for x in x0s_img]

# 为了 heatmap 在整段视频里颜色一致，固定 vmin/vmax
global_err_max = max(float(e.max()) for e in error_maps)
global_err_max = max(global_err_max, 1e-8)

# 如果你的 x0/target 已经是 samples=(latents/2+0.5).clamp(0,1) 处理后的结果，设 True 就行
x0_assume_image_range = True
target_assume_image_range = True

# target 只处理一次（固定显示）
target_disp = normalize_for_display(target_img, assume_image_range=target_assume_image_range)

# =========================
# 3) 画图 + 保存 MP4（对齐修复版）
# =========================
save_dir = "./vis"
os.makedirs(save_dir, exist_ok=True)
mp4_path = os.path.join(save_dir, f"FN2errors_x0_target_with_T_aligned_{plot_idx}2.mp4")

fps = 4

# 用 GridSpec 固定布局：左列放三张图，右列专门放 colorbar
fig = plt.figure(figsize=(7, 14))
gs = fig.add_gridspec(
    nrows=3, ncols=2,
    width_ratios=[20, 1],   # 左图宽，右侧 colorbar 窄
    height_ratios=[1, 1, 1],
    wspace=0.05,
    hspace=0.08
)

ax_err = fig.add_subplot(gs[0, 0])
cax    = fig.add_subplot(gs[0, 1])   # colorbar 专用轴
ax_x0  = fig.add_subplot(gs[1, 0])
ax_tgt = fig.add_subplot(gs[2, 0])

# 右下两个空格隐藏（保持列宽一致）
ax_dummy1 = fig.add_subplot(gs[1, 1]); ax_dummy1.axis("off")
ax_dummy2 = fig.add_subplot(gs[2, 1]); ax_dummy2.axis("off")

writer = FFMpegWriter(fps=fps)

with writer.saving(fig, mp4_path, dpi=150):
    for i, (x_img, err_map) in enumerate(zip(x0s_img, error_maps)):
        # 当前帧对应的 T
        T_val = to_scalar(T_list[i])
        mae_val = to_scalar(mae[i])
        mse_val = to_scalar(mse[i])
        T_str = format_T(T_val, int_digits=4, frac_digits=3)
        mae_str = format_T(mae_val, int_digits=0, frac_digits=5)
        mse_str = format_T(mse_val, int_digits=0, frac_digits=5)

        # 清空上一帧
        ax_err.clear()
        ax_x0.clear()
        ax_tgt.clear()
        cax.clear()

        fig.suptitle(f"Frame {i} | T = {T_str}, MSE = {mse_str}, MAE = {mae_str}", fontsize=12, y=0.995)

        # ===== 第1行：error heatmap =====
        im0 = ax_err.imshow(
            err_map,
            cmap="hot",
            vmin=0.0,
            vmax=global_err_max,
            aspect="equal"   # 如果你希望强制填满改成 "auto"
        )
        ax_err.set_title("Error Heatmap", fontsize=10)
        ax_err.axis("off")

        cbar = fig.colorbar(im0, cax=cax)
        cbar.set_label("Error", fontsize=9)
        cbar.ax.tick_params(labelsize=8)

        # ===== 第2行：x0 图像 =====
        x_disp = normalize_for_display(x_img, assume_image_range=x0_assume_image_range)
        if x_disp.ndim == 2:
            ax_x0.imshow(x_disp, cmap="gray", vmin=0, vmax=1, aspect="equal")
        else:
            ax_x0.imshow(x_disp, aspect="equal")
        ax_x0.set_title("x0", fontsize=10)
        ax_x0.axis("off")

        # ===== 第3行：target 图像（固定） =====
        if target_disp.ndim == 2:
            ax_tgt.imshow(target_disp, cmap="gray", vmin=0, vmax=1, aspect="equal")
        else:
            ax_tgt.imshow(target_disp, aspect="equal")
        ax_tgt.set_title("target", fontsize=10)
        ax_tgt.axis("off")

        writer.grab_frame()

plt.close(fig)
print(f"Saved MP4 to: {mp4_path}")

Saved MP4 to: ./vis/FN2errors_x0_target_with_T_aligned_02.mp4


In [8]:
def T_to_t_linear(Timesteps, t, T):
    T_end = Timesteps[-1]
    T_start = Timesteps[0]
    t_end = t[0]
    t_start = t[-2]
    k = (T_end - T_start) / (t_end - t_start)
    b = T_start - t_start * k
    return (T-b)/k
Timesteps = [719.0000, 750.1111, 781.2222, 812.3333, 843.4444, 874.5556, 905.6667,
    936.7778, 967.8889, 999.0000]
t = [0.0000, 0.1110, 0.2220, 0.3330, 0.4440, 0.5550, 0.6660, 0.7770, 0.8880,
    0.9990, 1.0000]
t = T_to_t_linear(Timesteps, t, 781.2222)
t

0.7770000792857141

In [ ]:
a = Timesteps - 781.222 > 0.01

In [ ]:
def T_to_t_continuous(Timesteps, t, T, stage_idx, clamp=True):
    T = float(T)

    stage_T_start = float(Timesteps[0].item())
    stage_T_end   = float(Timesteps[-1].item())

    t_start = float(self.t_window_per_stage[stage_idx][0].item())
    t_end   = float(self.t_window_per_stage[stage_idx][-1].item())

    if clamp:
        T = max(min(T, stage_T_end), stage_T_start)
    else:
        assert stage_T_start <= T <= stage_T_end, \
            f"T={T} out of stage range [{stage_T_start}, {stage_T_end}]"

    # 反解线性映射
    t = t_start + (T - stage_T_start) * (t_end - t_start) / (stage_T_end - stage_T_start)
    return t